# Can a small network find defaults better than predicting no default?

A bank wants to prioritize clients for further review rather than simply predict repayment for everyone. Using the historical UCI Default of Credit Card Clients dataset through OpenML, you will inspect the target, compare two matched networks using validation loss, and report one final test comparison. You will also save the complete prediction path and verify that reloading preserves its probabilities.

**How this lab works.** Section numbers match the lesson page. In each section: **Run** the cells, **Predict** before you run (write your guess in the `guess(...)` cell), **Modify** one thing and see what changes, then **Make** the small piece marked `# YOUR CODE`. A ✓ means it works. The last cell prints a hand-in line for the lesson page.

In [ ]:
# 0. Setup — run this first (Runtime → Run all also works)
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")
import json, base64
import numpy as np
import keras
import tensorflow as tf

print("TensorFlow", tf.__version__, "| Keras", keras.__version__,
      "| GPU:", bool(tf.config.list_physical_devices("GPU")))
if not keras.__version__.startswith("3."):
    raise RuntimeError("This lab needs Keras 3. In Colab: Runtime → Restart session, then run again.")
keras.utils.set_random_seed(42)      # same data split and starting weights every run

LESSON = "m1-7-keras-workflow-your-first"
_RESULTS, _GUESSES = {}, {}

def guess(key, value):
    """Write your prediction BEFORE running the next cell. Changing it afterwards is allowed but recorded."""
    _GUESSES[key] = value
    print(f"Guess saved: {key} = {value!r}. Now run the next cell and compare.")

def record(key, value):
    """Store a result for the hand-in at the end."""
    _RESULTS[key] = round(float(value), 6)
    return value

def check(name, ok, hint=""):
    """✓ when your code does what the section asks, ✗ with a hint when it does not."""
    print(("✓ " if ok else "✗ ") + name + ("" if ok else " — " + hint))
    return bool(ok)

def handin():
    payload = {"v": 1, "lesson": LESSON, "results": _RESULTS, "guesses": _GUESSES}
    code = base64.urlsafe_b64encode(json.dumps(payload, separators=(",", ":")).encode()).decode()
    print("Copy this whole line into the lesson page (Debrief → Hand-in):")
    print("HANDIN-1." + code)

## 1. 1. What would count as finding defaults?

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, accuracy_score, recall_score
import matplotlib.pyplot as plt

data = fetch_openml(data_id=42477, as_frame=True, parser="auto")
features = data.data.copy()
id_columns = [c for c in features.columns if str(c).lower() == "id"]
features = features.drop(columns=id_columns)
raw_target = data.target.astype(str)
print("Predictors:", features.columns.tolist())
print("Target name:", data.target.name, "Observed labels:", raw_target.unique().tolist())
print("Observed class counts:\n", raw_target.value_counts())
print("Actual first row:\n", features.iloc[0].to_string(), "\nTarget:", raw_target.iloc[0])
print("Table shape:", features.shape, "vs required (30000, 23); removed identifiers:", id_columns)

The UCI documentation defines target **0 as no default** and **1 as default**; this OpenML table calls the target `y`. Map those meanings explicitly rather than trusting alphabetical order. Finish each Make cell before continuing: later cells depend on its results.

In [ ]:
# Can always predicting no default exceed 75% accuracy while finding zero defaults?
# Enter "yes" or "no" before calculating the answer.
guess("majority_can_miss_defaults", "yes")

In [ ]:
# YOUR CODE: explicitly map documented labels '0' and '1' to integers 0 and 1.
target_mapping = None
# YOUR CODE: map raw_target and convert it to an int32 numpy array.
y = None
# YOUR CODE: calculate the accuracy and recall of always predicting no default.
whole_accuracy = None
whole_recall = None
# YOUR CODE: assert 30000 rows, 23 predictors, 6636 defaults and 23364 nondefaults.
check("target_and_baseline", y is not None and np.array_equal(y, raw_target.map({"0": 0, "1": 1}).to_numpy()) and whole_accuracy == 23364 / 30000 and whole_recall == 0, "Map documented 0 to no default and 1 to default. Majority accuracy is the nondefault share; a rule detecting no defaults has recall 0.")

In [ ]:
# Try row 100 instead. Inspect another client without changing the target mapping.
row_number = 0
print(features.iloc[row_number].to_string())
print("Recorded default:", y[row_number], "(0 = no default; 1 = default)")
print(f"Whole-table majority accuracy {whole_accuracy:.4f}; default recall {whole_recall:.4f} (both range 0–1)")
print("Descriptive whole-table scores are not substitutes for same-test baselines.")

## 2. 2. Split before learning the preparation.

Reserve test rows first, then take validation rows from the remainder, as in section 2. Neither Dense training nor normalization adaptation may use reserved rows. All category codes remain numeric in this first model; numerical distance between category codes is a disclosed limitation, not a general modeling recommendation.

In [ ]:
X = features.to_numpy(dtype="float32")
x_development, x_test, y_development, y_test = train_test_split(
    X, y, test_size=0.2, random_state=17, stratify=y)
x_train, x_val, y_train, y_val = train_test_split(
    x_development, y_development, test_size=0.25, random_state=17, stratify=y_development)
assert (len(x_train), len(x_val), len(x_test)) == (18000, 6000, 6000)
feature_order = features.columns.tolist()
print("Train / validation / test shapes:", x_train.shape, x_val.shape, x_test.shape)
print("Required row counts: 18000 / 6000 / 6000; required width: 23; dtype:", x_train.dtype)
print(f"Training default prevalence {y_train.mean():.4f} (range 0–1)")

In [ ]:
def score_rows(targets, probabilities):
    p = np.asarray(probabilities).reshape(-1)
    hard_predictions = p > 0.5
    return {"loss": float(log_loss(targets, p, labels=[0, 1])),
            "accuracy": float(accuracy_score(targets, hard_predictions)),
            "default_recall": float(recall_score(targets, hard_predictions, zero_division=0))}

training_prevalence = float(y_train.mean())
val_baseline = score_rows(y_val, np.full(len(y_val), training_prevalence))
record("validation_baseline_loss", val_baseline["loss"])
print(f"Validation prevalence BCE {val_baseline['loss']:.4f} vs uniform BCE {np.log(2):.4f}; lower is better, minimum 0")
print(f"Validation majority accuracy {val_baseline['accuracy']:.4f}; default recall {val_baseline['default_recall']:.4f} (both range 0–1)")

In [ ]:
norm = keras.layers.Normalization(axis=-1, name="training_preparation")
# YOUR CODE: assert that train, validation and test each have 23 predictor columns.
# YOUR CODE: adapt norm using training inputs only.
check("training_only_preparation", norm.built and all(a.shape[1] == 23 for a in (x_train, x_val, x_test)) and np.allclose(norm.mean.numpy().reshape(-1), x_train.mean(axis=0), rtol=1e-4, atol=1e-4), "Assert all three feature widths, then call norm.adapt(x_train), never adapt on validation or test rows.")

In [ ]:
# Try training row 50. Change the borrower, not the normalization references.
training_row = 0
prepared_row = norm(x_train[training_row:training_row + 1]).numpy()[0]
for name, raw, prepared in zip(feature_order[:6], x_train[training_row, :6], prepared_row[:6]):
    print(f"{name}: raw {raw:.2f}; normalized {prepared:.3f} (training mean 0, standard deviation 1)")
mean_error = float(np.max(np.abs(norm(x_train).numpy().mean(axis=0))))
print(f"Largest absolute normalized training mean {mean_error:.2e} vs ideal 0")

## 3. 3. Build the network and its learning contract.

The bank network is **23 raw inputs → saved normalization → 16 ReLU units → 1 sigmoid probability**. Its Dense layers have `23×16+16 + 16×1+1 = 401` trainable parameters; normalization stores separate non-trainable state. Each fresh candidate adapts its own normalization layer on the identical training array, and we verify that the resulting references match.

In [ ]:
# Count Dense weights and biases, excluding normalization state.
guess("dense_parameters", 0)
# Does compile itself change the Dense arrays? Enter "yes" or "no".
guess("compile_changes_weights", "no")

In [ ]:
def build_bank(drop_rate):
    inputs = keras.Input(shape=(23,), name="borrower")
    preparation = keras.layers.Normalization(axis=-1, name="normalization")
    preparation.adapt(x_train)
    prepared = preparation(inputs)
    hidden = keras.layers.Dense(16, activation="relu", name="hidden")(prepared)
    if drop_rate:
        hidden = keras.layers.Dropout(drop_rate, seed=17, name="dropout")(hidden)
    outputs = keras.layers.Dense(1, activation="sigmoid", name="default_probability")(hidden)
    return keras.Model(inputs, outputs)
keras.utils.set_random_seed(17)
model = build_bank(0.0)
initial_dense = {name: [w.copy() for w in model.get_layer(name).get_weights()] for name in ("hidden", "default_probability")}
np.testing.assert_allclose(model.get_layer("normalization")(x_val[:8]), norm(x_val[:8]), rtol=1e-6, atol=1e-6)
model.summary()

In [ ]:
def compile_bank(net):
    # YOUR CODE: compile net with fresh Adam(0.001), probability-based binary cross-entropy,
    # BinaryAccuracy named accuracy at 0.5, and Recall named default_recall at 0.5.
    return None
compile_bank(model)
check("learning_contract", isinstance(getattr(model, "loss", None), keras.losses.BinaryCrossentropy) and not model.loss.from_logits and all(np.array_equal(a, b) for name in initial_dense for a, b in zip(initial_dense[name], model.get_layer(name).get_weights())), "Compile with fresh Adam(0.001), probability-based BinaryCrossentropy, and accuracy/recall at 0.5; compile configures learning without changing Dense arrays.")

In [ ]:
trainable_count = int(sum(np.prod(w.shape) for w in model.trainable_weights))
state_count = int(sum(np.prod(w.shape) for w in model.non_trainable_weights))
print(f"Dense trainable parameters {trainable_count} vs calculated 401; normalization state {state_count} vs 47")
record("dense_parameters", trainable_count)
toy_inputs = keras.Input(shape=(2,))
toy_norm = keras.layers.Normalization(mean=[150000.0, 0.0], variance=[1e10, 1.0])
toy_prepared = toy_norm(toy_inputs)
toy_hidden_layer = keras.layers.Dense(2, activation="relu")
toy_hidden = toy_hidden_layer(toy_prepared)
toy_output_layer = keras.layers.Dense(1, activation="sigmoid")
toy_outputs = toy_output_layer(toy_hidden)
toy = keras.Model(toy_inputs, toy_outputs)
toy_hidden_layer.set_weights([np.array([[0.8, -0.6], [0.4, 0.2]], dtype="float32"), np.array([-0.2, 0.1], dtype="float32")])
toy_output_layer.set_weights([np.array([[1.2], [-0.5]], dtype="float32"), np.array([-0.4], dtype="float32")])
print("Miniature architecture: 2 normalized inputs → 2 ReLU units → 1 sigmoid; Dense count 9 vs 401 in the bank model")

In [ ]:
original_toy_row = np.array([[200000, 2]], dtype="float32")
toy_probability = float(toy(original_toy_row, training=False).numpy()[0, 0])
toy_loss = float(keras.losses.BinaryCrossentropy()(np.array([[1.0]], dtype="float32"), toy(original_toy_row)))
print(f"Assigned-weight probability {toy_probability:.4f} vs trace 0.6682; default BCE {toy_loss:.4f} vs uniform 0.6931")
record("toy_probability", toy_probability)
assert abs(toy_probability - 0.6681878) < 1e-5 and abs(toy_loss - 0.403186) < 1e-5
# Try 250000 instead. Keep delay, references and weights fixed; look for input sensitivity.
credit_limit = 200000
hypothetical = np.array([[credit_limit, 2]], dtype="float32")
p_hypothetical = float(toy(hypothetical, training=False).numpy()[0, 0])
print(f"Hypothetical probability {p_hypothetical:.4f} vs original {toy_probability:.4f} (range 0–1)")
print("Changing inputs is not retraining, causality, or evidence of calibration.")

## 4. 4. Fit means running the learning loop.

Define the callback before `fit`, even though section 5 examines its behavior in detail. For both candidates, reset the training seed to 17 immediately before fitting, use identical initial Dense arrays, and create fresh Adam and callback objects. This is a matched classroom comparison, not proof of statistical superiority; dropout also changes random-number consumption.

In [ ]:
def new_stopping_rule():
    return keras.callbacks.EarlyStopping(
        monitor="val_loss", mode="min", min_delta=0, patience=3,
        restore_best_weights=True)

early_stop = new_stopping_rule()
print("Stopping contract: minimize validation BCE; patience 3; epoch cap 30; restore best weights.")

In [ ]:
# Must the epoch with the lowest training loss also have the lowest validation loss?
# Enter "yes" or "no" before fitting.
guess("same_best_epoch", "no")

In [ ]:
keras.utils.set_random_seed(17)
history = model.fit(
    x_train, y_train,
    batch_size=128, epochs=30,
    validation_data=(x_val, y_val), validation_batch_size=128,
    callbacks=[early_stop], verbose=0)
training_losses = history.history["loss"]
validation_losses = history.history["val_loss"]
print(f"Executed {len(training_losses)} epochs vs cap 30; {int(np.ceil(len(x_train) / 128))} updates per epoch")
print(f"Last training BCE {training_losses[-1]:.4f}; last validation BCE {validation_losses[-1]:.4f} vs validation baseline {val_baseline['loss']:.4f}")

In [ ]:
# Try showing only the first 10 epochs. Change the view, not training or selection.
show_first_epochs = 30
n_visible = min(show_first_epochs, len(training_losses))
epoch_axis = np.arange(1, n_visible + 1)
plt.plot(epoch_axis, training_losses[:n_visible], label="Training BCE")
plt.plot(epoch_axis, validation_losses[:n_visible], label="Validation BCE")
plt.axhline(val_baseline["loss"], linestyle="--", color="gray", label="Same-validation prevalence baseline")
plt.xlabel("Epoch (one-based)"); plt.ylabel("Binary cross-entropy (lower is better)")
plt.legend(); plt.show()
print(f"Epoch {n_visible}: validation BCE {validation_losses[n_visible - 1]:.4f} vs baseline {val_baseline['loss']:.4f}")

## 5. 5. Keep the best validation weights, not simply the last weights.

A History entry describes the weights at that epoch; the current model instead contains restored best weights. With `min_delta=0`, the best validation epoch is the location of the minimum recorded validation loss. Re-evaluate using the same batch size and a small numerical tolerance, and report whether training stopped early or reached its cap.

In [ ]:
# Practice history only: 0.56, 0.53, 0.51, 0.515, 0.52.
# With patience 2 and min_delta 0, at which one-based epoch does training stop?
guess("illustrative_stop_epoch", 0)

In [ ]:
# YOUR CODE: find the one-based epoch with minimum measured validation loss.
best_epoch = None
epochs_run = len(validation_losses)
callback_stop_epoch = int(early_stop.stopped_epoch) + 1 if early_stop.stopped_epoch > 0 else 0
plain_validation = model.evaluate(x_val, y_val, batch_size=128, return_dict=True, verbose=0)
print(f"Best validation epoch {best_epoch} within executed range 1–{epochs_run}")
print("Callback stopping epoch:", callback_stop_epoch, "(0 means training reached the cap)")
print(f"Restored validation BCE {plain_validation['loss']:.4f} vs minimum history {min(validation_losses):.4f} and baseline {val_baseline['loss']:.4f}")
check("best_weights", best_epoch is not None and best_epoch == int(np.argmin(validation_losses)) + 1 and abs(plain_validation["loss"] - min(validation_losses)) < 1e-4, "argmin is zero-based: add 1 for the displayed epoch. Restored validation BCE should match the minimum within numerical tolerance, using the same evaluation batch size.")

In [ ]:
# Try patience 3 instead of 2, on this illustrative history only. Do not refit the bank model.
practice_patience = 2
practice_losses = [0.56, 0.53, 0.51, 0.515, 0.52, 0.525]
practice_best, wait, practice_stop = float("inf"), 0, None
for epoch, loss in enumerate(practice_losses, start=1):
    if loss < practice_best:
        practice_best, wait = loss, 0
    else:
        wait += 1
    if wait >= practice_patience:
        practice_stop = epoch
        break
print(f"Illustrative stop epoch {practice_stop} within range 1–6; best BCE {practice_best:.3f} vs stopping BCE {loss:.3f}; not a UCI result")
record("plain_best_epoch", best_epoch)
record("plain_epochs_run", epochs_run)

## 6. 6. Dropout is a training condition to test.

Fit exactly one fresh alternative: dropout rate **0.2** after the hidden ReLU layer. Keep training-only normalization references, initial Dense arrays, Adam settings, splits, batch size, stopping rule and epoch cap fixed. Select lower best validation BCE, with ties going to no dropout; no test score participates.

In [ ]:
# Must dropout improve validation BCE? Enter "yes" or "no".
guess("dropout_must_help", "no")
# For this particular run, which candidate do you expect to win: "plain" or "dropout"?
guess("expected_winner", "plain")

In [ ]:
keras.utils.set_random_seed(17)
dropout_model = build_bank(0.2)
for name, arrays in initial_dense.items():
    dropout_model.get_layer(name).set_weights(arrays)
compile_bank(dropout_model)
dropout_stop = new_stopping_rule()
assert all(np.array_equal(a, b) for name in initial_dense for a, b in zip(initial_dense[name], dropout_model.get_layer(name).get_weights()))
np.testing.assert_allclose(model.get_layer("normalization")(x_val[:8]), dropout_model.get_layer("normalization")(x_val[:8]), rtol=1e-6, atol=1e-6)
keras.utils.set_random_seed(17)
dropout_history = dropout_model.fit(
    x_train, y_train, batch_size=128, epochs=30,
    validation_data=(x_val, y_val), validation_batch_size=128,
    callbacks=[dropout_stop], verbose=0)
dropout_validation = dropout_model.evaluate(x_val, y_val, batch_size=128, return_dict=True, verbose=0)
print("Matched starting Dense arrays and normalization outputs verified; dropout rate 0.2 vs plain rate 0.0")

In [ ]:
dropout_model.summary()
histories = {"plain": history, "dropout": dropout_history}
validation_reports = {"plain": plain_validation, "dropout": dropout_validation}
for name, measured in histories.items():
    losses = measured.history["val_loss"]
    report = validation_reports[name]
    print(f"{name}: best epoch {np.argmin(losses) + 1} of {len(losses)}; best BCE {min(losses):.4f} vs baseline {val_baseline['loss']:.4f}")
    print(f"  Restored BCE {report['loss']:.4f}; accuracy {report['accuracy']:.4f} vs {val_baseline['accuracy']:.4f}; recall {report['default_recall']:.4f} vs {val_baseline['default_recall']:.4f}")
    assert abs(report["loss"] - min(losses)) < 1e-4, "Restored loss must match the best history loss within numerical tolerance."
    epochs = np.arange(1, len(losses) + 1)
    plt.plot(epochs, measured.history["loss"], linestyle="--", label=f"{name} training")
    plt.plot(epochs, losses, label=f"{name} validation")
plt.axhline(val_baseline["loss"], color="gray", linestyle=":", label="Validation baseline")
plt.xlabel("Epoch"); plt.ylabel("Binary cross-entropy (minimum 0)"); plt.legend(); plt.show()

In [ ]:
plain_best_loss = float(min(validation_losses))
dropout_best_loss = float(min(dropout_history.history["val_loss"]))
# YOUR CODE: choose the lower best validation BCE; ties must select plain.
selected_name = None
# YOUR CODE: assign the corresponding fitted model, without inspecting test scores.
selected_model = None
check("validation_only_choice", selected_name is not None and selected_name == ("dropout" if dropout_best_loss < plain_best_loss else "plain") and selected_model is (dropout_model if selected_name == "dropout" else model), "Compare the two minimum validation BCE values only; use strict less-than so a tie selects plain. Never use test scores for this choice.")

In [ ]:
selected_history = histories[selected_name]
selected_best_loss = float(min(selected_history.history["val_loss"]))
selected_best_epoch = int(np.argmin(selected_history.history["val_loss"])) + 1
record("selected_validation_loss", selected_best_loss)
record("selected_best_epoch", selected_best_epoch)
record("selected_dropout_rate", 0.2 if selected_name == "dropout" else 0.0)
print(f"Frozen choice {selected_name}: best validation BCE {selected_best_loss:.4f} vs baseline {val_baseline['loss']:.4f}")
activations = np.array([[1.0, 0.2]], dtype="float32")
scaled_activations = activations * np.array([[1.0, 0.0]], dtype="float32") / (1 - 0.5)
masked_probability = float(toy_output_layer(scaled_activations).numpy()[0, 0])
drop_demo = keras.layers.Dropout(0.5, seed=17)
np.testing.assert_allclose(drop_demo(activations, training=False).numpy(), activations)
np.testing.assert_allclose(toy(original_toy_row, training=False), toy(original_toy_row, training=False))
print(f"Specified-mask probability {masked_probability:.4f} vs unmasked {toy_probability:.4f}; default BCE {-np.log(masked_probability):.4f} vs unmasked {toy_loss:.4f}")
check("specified_mask", abs(masked_probability - 0.880797) < 1e-5, "The specified mask gives activations [2, 0]; ordinary inference leaves activations unchanged. One favorable mask is not evidence of better generalization.")

## 7. 7. Freeze, evaluate and save.

The candidate is now frozen; open the test branch once and compare all three scores with baselines on those same rows. The finite-loss baseline predicts training prevalence, while its hard predictions are all no default because that prevalence is below 0.5. A promising historical classroom result can justify investigating a review-prioritization signal, not an automatic lending decision.

In [ ]:
# Will ordinary inference after reload activate dropout or relearn normalization?
# Enter "neither", "dropout", "normalization", or "both".
guess("reload_inference_behavior", "neither")

In [ ]:
# YOUR CODE: evaluate the frozen selected_model once on x_test and y_test,
# using batch_size=128, requesting a dictionary and suppressing batch progress.
test_report = None
# YOUR CODE: use score_rows with y_test and one training_prevalence probability per test row.
test_baseline = None
check("final_evaluation", test_report is not None and test_baseline is not None and all(k in test_report for k in ("loss", "accuracy", "default_recall")) and abs(test_baseline["accuracy"] - float((y_test == 0).mean())) < 1e-12 and test_baseline["default_recall"] == 0 and abs(test_baseline["loss"] - log_loss(y_test, np.full(len(y_test), training_prevalence), labels=[0, 1])) < 1e-12, "Evaluate the frozen model once on test with targets; score a constant TRAINING prevalence on those same test rows for the finite-loss baseline.")

In [ ]:
print("Final test report for frozen candidate:", selected_name)
for metric in ("loss", "accuracy", "default_recall"):
    result, reference = test_report[metric], test_baseline[metric]
    improvement = reference - result if metric == "loss" else result - reference
    print(f"Test {metric}: model {result:.4f} vs same-test baseline {reference:.4f}; improvement {improvement:+.4f}")
print("Accuracy and recall range 0–1; BCE has minimum 0 and no finite upper bound.")
print(f"Selected best validation BCE {selected_best_loss:.4f} vs same-validation baseline {val_baseline['loss']:.4f}")
record("test_baseline_accuracy", test_baseline["accuracy"])
record("test_baseline_loss", test_baseline["loss"])
record("test_baseline_recall", test_baseline["default_recall"])
record("test_loss", test_report["loss"])
record("test_accuracy", test_report["accuracy"])
record("test_default_recall", test_report["default_recall"])
print("Default recall = defaults detected / actual defaults, not detected defaults / all clients.")

In [ ]:
fixed_validation_sample = x_val[:8].copy()
save_path = "bank_default_selected.keras"
before_reload = selected_model.predict(fixed_validation_sample, verbose=0)
selected_model.save(save_path)
reloaded_model = keras.models.load_model(save_path)
after_reload = reloaded_model.predict(fixed_validation_sample, verbose=0)
reload_difference = float(np.max(np.abs(before_reload - after_reload)))
np.testing.assert_allclose(before_reload, after_reload, rtol=1e-6, atol=1e-7)
np.testing.assert_equal(int(selected_model.optimizer.iterations.numpy()), int(reloaded_model.optimizer.iterations.numpy()))
print(f"Largest validation-sample probability change after reload {reload_difference:.2e} vs tolerance 1e-6")
print("Saved: architecture, Dense arrays, normalization state and compiled optimizer state.")
record("reload_max_difference", reload_difference)

In [ ]:
# Try validation row 25 instead. Change only the inference row, not the model or threshold.
inference_row = 0
borrower = x_val[inference_row:inference_row + 1]
probability = float(reloaded_model(borrower, training=False).numpy()[0, 0])
print(f"Validation borrower's default probability {probability:.4f} (range 0–1); review flag {probability > 0.5} at fixed threshold 0.5")
print("This classroom flag is not an automatic credit decision; no adaptation or fitting occurred.")

In [ ]:
# Repair this proposal: adapt on all rows, reuse Adam for a fresh candidate,
# choose dropout using test accuracy, and call the resulting flag an automatic decision.
# YOUR CODE: fill a dictionary with adapt_on, choose_on, fresh_optimizer,
# fresh_callback, test_for_selection, and threshold.
repairs = None
# YOUR CODE: write a limitations sentence mentioning historical data and 'no automatic' decisions.
limitations = None
check("workflow_repair", repairs is not None and repairs == {"adapt_on": "train", "choose_on": "validation", "fresh_optimizer": True, "fresh_callback": True, "test_for_selection": False, "threshold": 0.5} and isinstance(limitations, str) and "historical" in limitations.lower() and "no automatic" in limitations.lower(), "Learn preparation on training rows, select on validation BCE with fresh optimizer/callback objects, keep test out of selection, and state historical-data and no-automatic-decision limitations.")

## Hand-in

In [ ]:
# Hand-in — run this last, then paste the line it prints into the lesson page
handin()